# Adversarial Stress-Testing: The Corporate "Turing Test"

**Monte Carlo Failure-Cascade Simulation with Strategic Adversary**

Post 5 of 6 — *Audit 2.0 in the Age of Non-Deterministic Systems*
— Francesco Orsi · https://kunskap.substack.com

---

> *Instead of asking "are we safe?", run 10,000 simulations of "how could this company fail?" — then let the topology tell you where.*

**Two adversarial modes:**
1. **Scenario Monte Carlo** — 10,000 sims × 6 plausible shock profiles
2. **Strategic Adversary** — A synthetic agent that *optimally* selects which nodes to attack

**The audit finding:** if a single person or API appears in 90%+ of major failures, that is structural fragility — not bad luck.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.graph import build_graph, get_layout
from src.scenarios import get_scenarios
from src.cascade import CascadeEngine
from src.adversary import StrategicAdversary
from src.metrics import compute_bottleneck_rates, compute_adversary_frequency, compute_composite_threat
from src.plotting import (apply_style, fig1_graph, fig2_heatmap, fig3_ridgeplot,
                          fig4_bottleneck, fig5_adversary, fig6_cascade,
                          fig7_cofailure, fig8_dotstrip, fig9_threatmap)

import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

apply_style()
%matplotlib inline

G = build_graph()
pos = get_layout(G)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Fig. 1 — The Enterprise Dependency Graph

In [ ]:
fig1_graph(G, pos, outdir=".")
from IPython.display import Image
Image("fig1_graph.png")

## Scenario Monte Carlo — 60,000 Simulated Futures

Six compound stress scenarios, each run 10,000 times:
- **Baseline**: Normal operations
- **Cyber + Key-Person**: Ransomware + IT admin absence
- **Supply + Regulatory**: API restriction + inspection
- **Data Integrity Crisis**: EBR audit trail gaps
- **Talent Exodus**: Key staff leave during ERP migration
- **Black Swan**: Pandemic + cyber + regulatory

In [ ]:
scenarios = get_scenarios()
engine = CascadeEngine(G, seed=42)
results = []

for sc in scenarios:
    r = engine.run_scenario(sc, N=10000)
    results.append(r)
    fracs = r["fracs"]
    print(f"  [{sc.short:6s}] mean={fracs.mean():.1%}  "
          f"P95={np.percentile(fracs, 95):.0%}  major={r['n_major']}")

## Fig. 3 — Damage Distributions (Ridge Plot)

In [ ]:
fig3_ridgeplot(results, outdir=".")
Image("fig3_ridgeplot.png")

## Fig. 2 — Fragility Heatmap

In [ ]:
fig2_heatmap(G, results, outdir=".")
Image("fig2_heatmap.png")

## Fig. 4 — Bottleneck Frequency: The Primary Audit Finding

> *If a single person or API is a bottleneck in 90% of failure simulations, that is your primary audit finding.*

In [ ]:
bn_rates = compute_bottleneck_rates(results, G)
stressed = [r for r in results if r["scenario"].short != "BASE"]
total_major = sum(r["n_major"] for r in stressed)

print(f"Total major failures (>25% damage): {total_major:,}")
for n, rate in sorted(bn_rates.items(), key=lambda x: -x[1])[:8]:
    print(f"  {n:18s} [{G.nodes[n]['category']:7s}]  {rate:.0%}")

fig4_bottleneck(G, bn_rates, total_major, outdir=".")
Image("fig4_bottleneck.png")

## Act II: The Strategic Adversary

A synthetic agent that asks: *Given budget k, which k nodes to knock out for maximum damage?*

- k ≤ 3: exhaustive search over all C(19, k) combinations
- k > 3: greedy optimization

In [ ]:
adversary = StrategicAdversary(G, seed=42)
budget_results = adversary.budget_curve(max_k=5, n_sims=800, stress=1.5)

## Fig. 5 — Adversary Damage Curve + Target Selection

In [ ]:
fig5_adversary(G, budget_results, outdir=".")
Image("fig5_adversary.png")

## Fig. 6 — Cascade Anatomy from Adversary k=5 Attack

In [ ]:
fig6_cascade(G, pos, budget_results, outdir=".")
Image("fig6_cascade.png")

## Fig. 7 — Co-Failure Network

In [ ]:
fig7_cofailure(G, pos, results, outdir=".")
Image("fig7_cofailure.png")

## Fig. 8 — Cross-Scenario Fragility Profile

In [ ]:
fig8_dotstrip(G, results, outdir=".")
Image("fig8_dotstrip.png")

## Fig. 9 — The Threat Map: Before and After 60,000 Simulations

> Your org chart told you who reports to whom.
> The threat map tells you where your company breaks.

In [ ]:
adv_freq = compute_adversary_frequency(budget_results, G)
threat = compute_composite_threat(G, results, budget_results, bn_rates, adv_freq)

fig9_threatmap(G, pos, results, threat, bn_rates, adv_freq, budget_results, outdir=".")
Image("fig9_threatmap.png")

## Complete Audit Findings

In [ ]:
print("=" * 60)
print(" AUDIT FINDINGS SUMMARY")
print("=" * 60)

print("\n--- A. Structural Bottlenecks ---")
for n, rate in sorted(bn_rates.items(), key=lambda x: -x[1])[:8]:
    print(f"  {n:18s} [{G.nodes[n]['category']:7s}]  in {rate:.0%} of major failures")

print("\n--- B. Strategic Adversary ---")
for r in budget_results:
    print(f"  k={r['k']}: {r['targets']} -> {r['damage']:.1%}")

print("\n--- C. Co-Failure Pairs (Black Swan) ---")
cf = results[-1]["cofail"]
N_bs = results[-1]["N"]
for (a, b), c in sorted(cf.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {a} + {b}: {c/N_bs:.1%}")

print("\n--- D. Composite Threat (Top 10) ---")
for n, t in sorted(threat.items(), key=lambda x: -x[1])[:10]:
    print(f"  {n:18s} [{G.nodes[n]['category']:7s}]  threat={t:.3f}")
print("=" * 60)